# Firing rate maps (per-trajectory, not pooled)

Computes occupancy and firing-rate maps for one neuron at a time, separately for each of the 50 trajectories.

Design notes (fixes vs. the original pipeline):
- Occupancy map and firing-rate map are built with the *same* `np.histogram2d` call signature (same `bins`, same fixed `range`), so they always land on an identical grid — no separate meshgrid/nearest-bin search, no row/column mismatch possible.
- Bin edges are pinned explicitly to the known trajectory bounds `[0, 1] x [0, 1]` via `range=`, instead of being derived from each call's own data min/max — every trajectory's map is on the exact same physical grid, so maps are directly comparable across trials.
- No `rotate()` calls anywhere. The only reorientation is a single `.T` + `origin='lower'` applied consistently at plot time (see `plot_random_trajectory_maps`), so there's nothing that can drift out of sync between maps.
- Activation values are converted to amplitude (`np.abs`) before binning, so complex-valued activity from the oscillatory network is handled explicitly rather than silently truncated by an implicit complex-to-real cast.
- Firing rate = occupancy-weighted mean amplitude per bin (a spatial average), not a threshold-then-peak value — avoids an arbitrary spike threshold and matches how rate maps are usually computed for continuous-valued unit activity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from scipy.ndimage import gaussian_filter

In [ ]:
# Trajectory grid bounds and map resolution.
# 1000 points per trajectory / 32x32 bins ~= 1 raw sample per bin before smoothing —
# fine enough to resolve field structure without every bin being empty.
BOUNDS = (0.0, 1.0)
N_BINS = 32
SIGMA = 1.0     # Gaussian smoothing bandwidth, in bins
EPS = 1e-10

### Input

In [ ]:
def get_input_data(activation_path, trajectory_path):
    '''
    Expects trajectory_path to unpickle to a dict with 'x','y' arrays of shape (n_traj, n_time),
    and activation_path to unpickle to an array of shape (n_traj, n_time, n_neurons).
    Returns positions (n_traj, n_time, 2) and activations (n_traj, n_time, n_neurons).
    '''
    with open(trajectory_path, 'rb') as f:
        data_pos = pickle.load(f)
    with open(activation_path, 'rb') as f:
        activations = pickle.load(f)

    x = np.asarray(data_pos['x'])
    y = np.asarray(data_pos['y'])
    positions = np.stack([x, y], axis=-1)
    activations = np.asarray(activations)
    return positions, activations

### Mapping functions

In [ ]:
def occupancy_map_func(positions, bins=N_BINS, bounds=BOUNDS, sigma=SIGMA):
    '''
    positions: (T, 2) single trajectory.
    Returns a smoothed occupancy-count map, shape (bins, bins), axis 0 = x-bin, axis 1 = y-bin.
    '''
    H, xedges, yedges = np.histogram2d(
        positions[:, 0], positions[:, 1],
        bins=bins, range=[bounds, bounds]
    )
    occupancy = gaussian_filter(H, sigma=sigma)
    return occupancy, xedges, yedges

In [ ]:
def firing_rate_map_func(positions, activity, occupancy_map, bins=N_BINS, bounds=BOUNDS, sigma=SIGMA, eps=EPS):
    '''
    positions: (T, 2) single trajectory.
    activity:  (T,) one neuron's activation for that same trajectory (real or complex).
    occupancy_map: output of occupancy_map_func for this same trajectory (same bins/bounds/sigma).
    Returns the occupancy-normalized firing-rate map, shape (bins, bins), same x/y-bin
    convention as occupancy_map since both come from the same histogram2d(bins=, range=) call.
    '''
    amplitude = np.abs(activity)

    activity_sum, _, _ = np.histogram2d(
        positions[:, 0], positions[:, 1],
        bins=bins, range=[bounds, bounds],
        weights=amplitude
    )
    activity_sum = gaussian_filter(activity_sum, sigma=sigma)

    rate_map = activity_sum / (occupancy_map + eps)
    return rate_map

In [ ]:
def firing_rate_maps_for_neuron(positions, activations, neuron_idx, bins=N_BINS, bounds=BOUNDS, sigma=SIGMA, eps=EPS):
    '''
    positions:   (n_traj, T, 2)
    activations: (n_traj, T, n_neurons)
    Computes one occupancy map + one firing-rate map per trajectory (not pooled).
    Returns firing_maps, occupancy_maps, each shape (n_traj, bins, bins).
    '''
    n_traj = positions.shape[0]
    firing_maps = np.zeros((n_traj, bins, bins))
    occupancy_maps = np.zeros((n_traj, bins, bins))

    for t in range(n_traj):
        occ_map, _, _ = occupancy_map_func(positions[t], bins=bins, bounds=bounds, sigma=sigma)
        fr_map = firing_rate_map_func(
            positions[t], activations[t, :, neuron_idx], occ_map,
            bins=bins, bounds=bounds, sigma=sigma, eps=eps
        )
        occupancy_maps[t] = occ_map
        firing_maps[t] = fr_map

    return firing_maps, occupancy_maps

### Plotting

In [ ]:
def plot_random_trajectory_maps(firing_maps, neuron_idx, n_show=6, bounds=BOUNDS, seed=None):
    '''
    Plots the firing-rate map for n_show randomly chosen trajectories, for one neuron.
    firing_maps: (n_traj, bins, bins) as returned by firing_rate_maps_for_neuron.
    '''
    rng = np.random.default_rng(seed)
    n_traj = firing_maps.shape[0]
    n_show = min(n_show, n_traj)
    chosen = rng.choice(n_traj, size=n_show, replace=False)

    ncols = min(3, n_show)
    nrows = int(np.ceil(n_show / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)

    for ax, traj_idx in zip(axes.ravel(), chosen):
        # firing_maps[traj_idx] is indexed [x_bin, y_bin]; transpose + origin='lower'
        # displays x horizontal and y vertical-increasing-upward, applied identically every time.
        im = ax.imshow(
            firing_maps[traj_idx].T, origin='lower',
            extent=[bounds[0], bounds[1], bounds[0], bounds[1]],
            cmap='jet', aspect='auto'
        )
        ax.set_title(f'Trial {traj_idx}')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        fig.colorbar(im, ax=ax, shrink=0.8)

    for ax in axes.ravel()[len(chosen):]:
        ax.axis('off')

    fig.suptitle(f'Neuron {neuron_idx} — firing rate maps ({n_show} random trials)')
    plt.tight_layout()
    plt.show()

### Example usage

In [ ]:
activation_path = '0_input_data/activations_z3_traj_5.pkl'   # update to your 50-trajectory file
trajectory_path = '0_input_data/traj_5_data.pkl'             # update to your 50-trajectory file

positions, activations = get_input_data(activation_path, trajectory_path)
print(positions.shape, activations.shape)   # expect (50, 1000, 2), (50, 1000, 50)

In [ ]:
neuron_idx = 0

firing_maps, occupancy_maps = firing_rate_maps_for_neuron(positions, activations, neuron_idx)

plot_random_trajectory_maps(firing_maps, neuron_idx, n_show=6, seed=0)